# 正常扩声现场 A/B（A 角色操作）

目的：在原反馈路径夹具和固定普通语音源下，比较同一PL增益的旁路/抵消，决定下一轮阈值复测还是交B检查模型。

现场准备：MAX4466在关盖箱内接LINE_IN，HP+MIC接有源音响，不戴耳机；箱外固定手机循环语音。位置/方向/音量、接线、MAX4466供电/增益及operator_notes填入field_setup.json。现场人员确认供电及接线，并准备立即切断音响电源。

最多8个增益级，每级旁路/抵消各2秒，最多16次、发声累计32秒；总流程预计5–10分钟，依实际档位和现场操作而变。完成标志是summary.json；最后只返回该JSON和现场配置说明，原始NPZ留板。

电脑本地配置模板、完整路径和上传方法见项目操作包。板卡路径为 `/home/xilinx/jupyter_notebooks/base/audio/audio_closed/`，Jupyter文件列表为 `base → audio → audio_closed`。配置不完整或不能确认原夹具时不启动。

本Notebook不会自动播放。先按代码格标题和首行运行代码格1确认软件包哈希，再运行代码格2一次。脚本每项都等待现场START；任意尖叫/增长立即切断音响电源并停止整批。结束后只返回 `field_results/field_*/summary.json`；原始NPZ留在板卡。
## 代码格1：软件包核验（静音）
为什么：核对运行版本。在哪里：本标题下代码格，首行为 `import hashlib, importlib, json`。怎么做：点击后按Shift+Enter一次。预期：打印 `FIELD_PACKAGE_OK; NO_AUDIO_STARTED`。下一步：成功后运行代码格2；失败时保存输出并停止。

In [ ]:
import hashlib, importlib, json
from pathlib import Path
import os
BOARD_AUDIO_DIR = Path('/home/xilinx/jupyter_notebooks/base/audio/audio_closed')
if not BOARD_AUDIO_DIR.is_dir():
    raise FileNotFoundError('板卡闭环目录不存在：' + str(BOARD_AUDIO_DIR) + '；先在Jupyter文件列表打开 base → audio → audio_closed')
os.chdir(str(BOARD_AUDIO_DIR))
manifest_path = BOARD_AUDIO_DIR / 'manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError('板卡缺少 manifest.json：' + str(manifest_path) + '；从电脑本地 board/audio_closed/manifest.json 上传到该目录后重跑本格')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
for name in ('audio_closed.bit', 'audio_closed.hwh', 'closed_trial.py', 'closed_tools.py', 'residual_tools.py', 'closed_batch.py'):
    expected = manifest['hashes']['board/audio_closed/' + name]
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == expected, name
field_script = BOARD_AUDIO_DIR / 'field_validation.py'
if not field_script.is_file():
    raise FileNotFoundError('板卡目录缺少 field_validation.py：' + str(field_script) + '；请从电脑本地 board/audio_closed/field_validation.py 上传到 base → audio → audio_closed，然后重跑本代码格1')
expected_field_hash = 'BB56D866518814CECEA344C334099C9400D557C12A521C3292C20FD1AFA19EFA'
actual_field_hash = hashlib.sha256(field_script.read_bytes()).hexdigest()
if actual_field_hash.lower() != expected_field_hash.lower():
    raise ValueError('板卡 field_validation.py 版本不匹配：实际SHA256=' + actual_field_hash + '；预期=' + expected_field_hash + '。请上传本地指定版本后重跑代码格1')
setup_path = BOARD_AUDIO_DIR / 'field_setup.json'
if not setup_path.is_file():
    raise FileNotFoundError('板卡目录缺少 field_setup.json：' + str(setup_path) + '；请从电脑本地 board/audio_closed/field_setup.json 上传到该目录后重跑本代码格1')
assert json.loads(setup_path.read_text(encoding='utf-8'))['schema_version'] == 1
import closed_trial, closed_batch, field_validation
importlib.reload(closed_trial); importlib.reload(closed_batch); importlib.reload(field_validation)
print('FIELD_PACKAGE_OK; NO_AUDIO_STARTED')

## 代码格2：运行一次配对批次

为什么：完成正常语音下的固定条件配对。在哪里：本标题下代码格，首行为 `field_result = field_validation.run_field_validation()`。怎么做：点击代码格，按Shift+Enter一次。预期：先静音检查和打印模型范围；输入READY确认现场，检查范围/哈希，再输入START确认整批；每项配置静音后输入START才播放2秒。继续：听感码0/1且数字核对通过才进入下一项。停止：2/3/4/q、算术/配置/饱和/恢复异常停止整批；持续尖叫立即断音响电源。模型估算不是实测稳定阈值。

In [ ]:
field_result = field_validation.run_field_validation()